# 05 · Multiple Datasets: GNSS + InSAR

*Combine complementary measurements without hiding their relative influence.*

**Learning objectives**

- derive line-of-sight projection from a look vector
- stack datasets and covariances consistently
- construct named GNSS and InSAR datasets
- audit joint fits with per-dataset diagnostics

**Prerequisites:** Chapter 04; covariance-weighted inversion  
**Estimated time:** 60–90 minutes

Notation follows the [glossary](../docs/glossary.md); axes, units, signs, and
ordering follow [GeoDef conventions](../docs/conventions.md).


## Motivation

GNSS measures several motion components accurately at sparse stations; InSAR
measures one line-of-sight component at many pixels. Their weaknesses differ,
so a joint inversion can identify slip that either dataset alone leaves
ambiguous. Joint inversion is only useful when projection, units, covariance,
and dataset identity are kept consistent.


## Theory deepening: projection, stacking, and balance

An InSAR pixel observes the dot product

$$d_{LOS}=u_E\ell_E+u_N\ell_N+u_U\ell_U,$$

where $\hat\ell$ is a declared ENU unit look vector. Heading and incidence
determine that vector, but product conventions differ; projecting a known
upward displacement is a useful sign check.

Joint inversion stacks $d=[d_1;d_2]$, $G=[G_1;G_2]$, and a block covariance.
Multiplying one dataset covariance by $a^2$ divides its weight by $a^2$.
Dataset-specific reduced chi-squared values reveal whether one input dominates
or contains systematic residuals. Dataset names preserve this partition in
`InversionResult`, avoiding manual row slicing.


## Why join datasets?

| | GNSS | InSAR |
| --- | --- | --- |
| sampling | sparse (few stations) | dense (many pixels) |
| components | full 3-D (E, N, U) | 1-D line-of-sight |
| reference | absolute | relative |

Their strengths are complementary: GNSS pins down the 3-D motion at a few points;
InSAR fills in the dense spatial pattern.

### The line-of-sight projection
A radar satellite measures only the component of ground motion **along its
viewing direction**. If the 3-D surface displacement is $\mathbf u = (u_e, u_n, u_u)$
and the unit look vector is $\hat{\mathbf l} = (l_e, l_n, l_u)$, the observation is the
scalar

$$ d_{\text{LOS}} = \mathbf u \cdot \hat{\mathbf l} = l_e u_e + l_n u_n + l_u u_u. $$

`geodef.InSAR` stores the per-pixel look vector and applies this projection when
building $G$, exactly as `GNSS` interleaves E/N/U rows.

### Stacking
A joint inversion just stacks the two forward problems vertically:

$$ \begin{bmatrix} G_{\text{GNSS}} \\ G_{\text{InSAR}} \end{bmatrix} \mathbf m
   = \begin{bmatrix} \mathbf d_{\text{GNSS}} \\ \mathbf d_{\text{InSAR}} \end{bmatrix}, $$

with a block-diagonal data covariance. Each dataset's own uncertainties set its
weight, so the more precise data naturally count more.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import geodef

%load_ext autoreload
%autoreload 2

rng = np.random.default_rng(0)

# --- Recurring teaching scenario -------------------------------------------
# The higher-resolution megathrust from notebook 01, reused (copied) across
# notebooks 03-10 so every inversion targets the same "true" slip model.
fault = geodef.Fault.planar(
    lat=-2.0, lon=100.0, depth=25e3,   # centroid 25 km deep
    strike=315.0, dip=15.0,            # NW-striking, shallow dip
    length=180e3, width=90e3,          # 180 km x 90 km
    n_length=12, n_width=6,            # -> 72 patches
)
N = fault.n_patches
nL, nW = fault.grid_shape

# "True" slip: notebook 01's smooth Gaussian thrust asperity, dip-slip only.
# The model vector is blocked as [strike-slip (N) | dip-slip (N)], so the
# strike-slip half is zero.
i = np.arange(N) % nL
j = np.arange(N) // nL
i0, j0 = (nL - 1) / 2, nW / 2
bump = np.exp(-(((i - i0) / 3.0) ** 2 + ((j - j0) / 1.5) ** 2))
slip_true = np.concatenate([np.zeros(N), 3.0 * bump])

# A grid of surface GNSS stations (note: GNSS takes lon, lat order).
glon, glat = np.meshgrid(np.linspace(98.5, 101.5, 8), np.linspace(-3.6, -0.4, 8))
glon, glat = glon.ravel(), glat.ravel()
n_sta = glon.size

# Synthetic data: forward-model the true slip, then add seeded Gaussian noise.
_zero = np.zeros(n_sta)
_one = np.ones(n_sta)
G_full = geodef.greens.matrix(
    fault, geodef.data.gnss(lon=glon, lat=glat, east=_zero, north=_zero, up=_zero, sigma_east=_one, sigma_north=_one, sigma_up=_one)
)
sigma = 0.01  # 1 cm station noise
d_obs = G_full @ slip_true + rng.normal(0.0, sigma, G_full.shape[0])
gnss = geodef.data.gnss(
    lon=glon, lat=glat,
    east=d_obs[0::3], north=d_obs[1::3], up=d_obs[2::3],
    sigma_east=np.full(n_sta, sigma), sigma_north=np.full(n_sta, sigma), sigma_up=np.full(n_sta, sigma),
)
print(f"{N} patches, {n_sta} stations, {d_obs.size} observations")

In [ ]:
# Build a synthetic InSAR track over the same fault and true slip.
# InSAR measures one number per pixel: ground motion projected onto the
# satellite line-of-sight (LOS) direction, l_hat = (look_e, look_n, look_u).
ilon, ilat = np.meshgrid(np.linspace(99.5, 100.5, 20), np.linspace(-0.4, 0.4, 20))
ilon, ilat = ilon.ravel(), ilat.ravel()
n_px = ilon.size

# A typical ascending-track look vector (mostly up, slightly east/north).
look_e = np.full(n_px, -0.38)
look_n = np.full(n_px, 0.09)
look_u = np.full(n_px, 0.92)

# Forward-model the true slip into LOS, then add correlated-free noise.
Gi = geodef.greens.matrix(
    fault, geodef.data.insar(lon=ilon, lat=ilat, los=np.zeros(n_px), sigma=np.ones(n_px),
                        look_e=look_e, look_n=look_n, look_u=look_u)
)
sigma_los = 0.005  # 5 mm
los = Gi @ slip_true + rng.normal(0.0, sigma_los, n_px)
insar = geodef.data.insar(lon=ilon, lat=ilat, los=los, sigma=np.full(n_px, sigma_los),
                     look_e=look_e, look_n=look_n, look_u=look_u)
print(f"InSAR track: {n_px} pixels")


## The two views of the same slip

GNSS vectors (left) and the InSAR LOS field (right). Note how few GNSS stations
there are compared with the dense InSAR pixels.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
geodef.plot.vectors(gnss, fault, ax=ax1, title="GNSS displacements")
geodef.plot.insar(insar, fault, ax=ax2, title="InSAR line-of-sight")
plt.tight_layout()

## Single-dataset vs. joint inversion

Invert GNSS alone, InSAR alone, and both together (same smoothing for all).
The joint solution combines the constraints.

The top row shows the recovered slip; the bottom row shows the error (recovered $-$ true) against the same truth on a shared diverging scale. GNSS-only and the joint solution track the truth closely, while InSAR alone -- a single line-of-sight component -- is badly under-constrained and its error dwarfs the others.

In [ ]:
kw = dict(components="dip", regularization="laplacian", regularization_strength=1.0)
r_gnss = geodef.solve(fault, gnss, **kw)
r_insar = geodef.solve(fault, insar, **kw)
r_joint = geodef.solve(fault, [gnss, insar], **kw)

truth = slip_true[N:]
vmax = truth.max()
panels = [("True", None), ("GNSS only", r_gnss),
          ("InSAR only", r_insar), ("Joint", r_joint)]
fig, axes = plt.subplots(2, len(panels), figsize=(3.2 * len(panels), 6),
                         constrained_layout=True)

# Row 0: true + each inversion, shared symmetric RdBu_r scale.
for ax, (name, r) in zip(axes[0], panels):
    m = truth if r is None else r.dip_slip
    geodef.plot.slip(fault, m, ax=ax, cmap="RdBu_r",
                     vmin=-vmax, vmax=vmax, colorbar=False, title=name)

# Row 1: recovered - true, so under-constrained InSAR-only stands out.
diffs = [(name, r.dip_slip - truth) for name, r in panels if r is not None]
dmax = max(np.abs(d).max() for _, d in diffs)
axes[1, 0].axis("off")
for ax, (name, d) in zip(axes[1, 1:], diffs):
    geodef.plot.slip(fault, d, ax=ax, cmap="RdBu_r",
                     vmin=-dmax, vmax=dmax, colorbar=False, title=f"{name} - true")

fig.colorbar(axes[0, 0].collections[-1], ax=axes[0, :], shrink=0.8,
             label="Slip (m)")
fig.colorbar(axes[1, 1].collections[-1], ax=axes[1, :], shrink=0.8,
             label="Recovered - true (m)")

## Per-dataset fit

For a joint inversion, `invert.diagnostics` reports the fit to each dataset
separately, so you can check that neither is being ignored.

In [ ]:
for name, diag in geodef.invert.diagnostics(r_joint).items():
    print(f"{name}: reduced chi^2 = {diag.reduced_chi2:.2f}  "
          f"({diag.n_obs} obs, WRMS {diag.wrms * 1000:.1f} mm)")

## Checkpoint questions

**Why is InSAR not a vertical displacement dataset?**

<details><summary>Answer</summary>



LOS combines East, North, and Up through the look vector.



</details>

**How does doubling all uncertainties in one dataset change its weight?**

<details><summary>Answer</summary>



Its covariance grows by four and precision falls by four.



</details>

**Why name datasets?**

<details><summary>Answer</summary>



Names preserve stable partitions for predictions, residuals, and diagnostics.



</details>


## Common mistakes

- Reversing a look vector reverses the modeled LOS sign.
- Mixing velocity and displacement in one solve gives meaningless units.
- Balancing row counts instead of covariance can let dense InSAR overwhelm GNSS.


## Recap

- GNSS and InSAR constrain complementary components and spatial scales.
- Projection and covariance define how each dataset enters the stack.
- Named diagnostics reveal imbalance without manual slices.

Return to the [workflow decision guides](../docs/workflow.md) when adapting
this method. The course map in [the tutorial README](README.md) identifies the
next chapter and optional branches.


## Exercises

Predict the qualitative outcome before running each experiment. Worked
solutions are published separately in `solutions/05_multiple_datasets_solutions.ipynb`.

1. Inflate InSAR sigma by factors of two and ten and track the solution.
2. Add a descending look vector and quantify the change in uncertainty.
3. Reverse one look vector and diagnose the residual pattern.
4. Challenge: reproduce the joint solution with explicit stacked observations and matrices.


## Further reading

- Hanssen (2001), InSAR measurement and atmospheric errors.
- Segall (2010), joint geodetic inverse problems.
- Aster, Borchers & Thurber (2018), weighted data combination.
